# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinivas25046/FlyRank-MLstarter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 4 — The Freshness Multiplier.** The paper reports that among pages older than a
year, the cohort refreshed in the last 30 days shows a **1.6x health lift** and a **52x
impression lift** versus pages last updated 181-360 days ago (82 to 4.2K impressions).

*My methodology question:* where does "refreshed" come from as a label — was it randomly
assigned, or is it a page a team *chose* to refresh? If refresh candidates are picked because
they already show early recovery signals (seasonal upcoming demand, a competitor stumbling,
incoming backlinks), the 52x gap partly reflects *which pages get chosen*, not what refreshing
*does* to a page. The paper's own methodology section names "confounding variables" as a limit
but doesn't say whether refresh selection was random or judgment-based for this specific
comparison. I'd ask: was refresh timing independent of a page's trajectory at selection time, and
if the 52x figure is a mean, does it hold at the median too, or is it a few very large pages
pulling the average?

**Growth Prediction and 30-Day Momentum models (Part IV).** The paper reports "Same Brand, New
Pages" vs. "Unseen Brands" accuracy for three models (Growth: 90%/75%, Zombie Recovery: 99%/97%,
Momentum: 95%/90%) — commendably testing both, the same instinct behind my own grouped-vs-random
split below.

*My methodology question:* what's the base rate for each label? A 90% same-brand accuracy on a
label that's naturally ~90% one class is a different finding than 90% on a 50/50 split — and I
couldn't find the base rate stated next to any of the three model results. I'd also ask what
specifically explains the same-brand-to-new-brand accuracy drop (Growth's 90% -> 75% is the
largest gap of the three) — is it something like a client-scale or vertical signal baked into a
feature, the same client-memorization risk my own grouped split below is built to catch?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Both questions above are about the paper's methodology and don't require re-running its
# analysis -- there's no code needed here beyond stating the questions, which live in the
# markdown cell above. (A code cell that has nothing to compute is itself honest -- forcing
# a query here would just be decoration.)
print("See markdown above for the two methodology questions.")

See markdown above for the two methodology questions.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

This is the same grouped-by-client vs. random-row-split comparison already built in
`w05_model.ipynb`, rebuilt self-contained here as the required before/after demonstration.
**Before (random split):** a naive 80/20 row split can let the model see 80% of a given client's
pages in training and get "tested" on the other 20% of the *same* client — quietly rewarding
memorization of that client's own baseline SEO maturity rather than a pattern that generalizes.
**After (grouped split):** `GroupShuffleSplit` by `client_hash_id` holds out entire clients, so
the test set only contains clients the model has never seen at all. The gap between the two
numbers below *is* the finding -- exactly the same instinct behind the paper's own
same-brand-vs-new-brand reporting in Section 1 above.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb scikit-learn

import duckdb
import numpy as np
import pandas as pd
from getpass import getpass
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Hugging Face READ token (from a Colab Secret named HF_TOKEN ideally): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{BASE}/dim_content.parquet"
DAILY_FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"
FEATURE_MONTH = "2026-02"
LABEL_MONTH = "2026-03"

feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS feb_impressions,
           SUM(gsc_clicks)      AS feb_clicks,
           AVG(gsc_avg_position) AS feb_avg_position,
           SUM(ga4_sessions)     AS feb_sessions,
           SUM(sessions_ai)      AS feb_ai_sessions,
           SUM(CASE WHEN report_date < DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h1,
           SUM(CASE WHEN report_date >= DATE '{FEATURE_MONTH}-15' THEN gsc_impressions ELSE 0 END) AS feb_h2
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{FEATURE_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

label = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS mar_impressions
    FROM read_parquet('{DAILY_FACT}', hive_partitioning=true)
    WHERE month = '{LABEL_MONTH}'
    GROUP BY client_hash_id, content_hash_id
""").df()

panel = feat.merge(label, on=["client_hash_id", "content_hash_id"], how="inner")
panel = panel[panel["feb_impressions"] > 0].copy()
panel["declined_next_month"] = (panel["mar_impressions"] < panel["feb_impressions"] * 0.8).astype(int)

content_meta = con.sql(f"SELECT content_hash_id, content_created_date FROM read_parquet('{DIM_CONTENT}')").df()
panel = panel.merge(content_meta, on="content_hash_id", how="left")
panel["content_created_date"] = pd.to_datetime(panel["content_created_date"])
as_of = pd.Timestamp(f"{FEATURE_MONTH}-28")
panel["age_days"] = (as_of - panel["content_created_date"]).dt.days
panel["declining_now"] = (panel["feb_h2"] < panel["feb_h1"] * 0.8).astype(int)

FEATURE_COLS = ["feb_impressions", "feb_clicks", "feb_avg_position", "feb_sessions", "feb_ai_sessions", "age_days", "declining_now"]
LABEL_COL = "declined_next_month"
model_df = panel.dropna(subset=FEATURE_COLS + [LABEL_COL]).copy()


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


def fit_and_score(train_df, test_df):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[FEATURE_COLS])
    X_test = scaler.transform(test_df[FEATURE_COLS])
    y_train, y_test = train_df[LABEL_COL].values, test_df[LABEL_COL].values
    logreg = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000).fit(X_train, y_train)
    scores = logreg.predict_proba(X_test)[:, 1]
    return precision_at_k(scores, y_test, 20), precision_at_k(scores, y_test, 50), y_test.mean()


# BEFORE: naive random row split
train_random, test_random = train_test_split(model_df, test_size=0.2, random_state=RANDOM_SEED)
overlap_random = set(train_random["client_hash_id"]) & set(test_random["client_hash_id"])
p20_r, p50_r, base_r = fit_and_score(train_random, test_random)

# AFTER: grouped by client
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
tr_idx, te_idx = next(gss.split(model_df, groups=model_df["client_hash_id"]))
train_grouped, test_grouped = model_df.iloc[tr_idx], model_df.iloc[te_idx]
overlap_grouped = set(train_grouped["client_hash_id"]) & set(test_grouped["client_hash_id"])
p20_g, p50_g, base_g = fit_and_score(train_grouped, test_grouped)

print("== BEFORE: random row split ==")
print(f"Overlapping clients between train/test: {len(overlap_random)} (risk this split doesn't control for)")
print(f"Precision@20: {p20_r:.1%}  |  Precision@50: {p50_r:.1%}  |  base rate: {base_r:.1%}")

print("\n== AFTER: grouped-by-client split ==")
print(f"Overlapping clients between train/test: {len(overlap_grouped)} (should be 0)")
print(f"Precision@20: {p20_g:.1%}  |  Precision@50: {p50_g:.1%}  |  base rate: {base_g:.1%}")

print(f"\nGap at precision@20 (random - grouped): {(p20_r - p20_g)*100:.1f} points")
print("-> A large positive gap here would mean the random-split number was flattering the model")
print("   with client-specific memorization, same instinct as the paper's own same-brand-vs-new-brand split.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

== BEFORE: random row split ==
Overlapping clients between train/test: 24 (risk this split doesn't control for)
Precision@20: 45.0%  |  Precision@50: 52.0%  |  base rate: 30.6%

== AFTER: grouped-by-client split ==
Overlapping clients between train/test: 0 (should be 0)
Precision@20: 85.0%  |  Precision@50: 78.0%  |  base rate: 40.0%

Gap at precision@20 (random - grouped): -40.0 points
-> A large positive gap here would mean the random-split number was flattering the model
   with client-specific memorization, same instinct as the paper's own same-brand-vs-new-brand split.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# The static column-set check from w04/w05 (score/feature inputs never touch the label or a
# future window) already passed by construction. Here's the DYNAMIC version the
# hunting-leakage-and-validating skill actually asks for: deliberately ADD a leaky feature and
# watch the score jump toward 1.0 -- if it doesn't jump, the test harness itself is broken.

leak_df = model_df.copy()
leak_df["mar_impressions"] = panel.loc[leak_df.index, "mar_impressions"]  # the literal future/label-source column

LEAKY_FEATURE_COLS = FEATURE_COLS + ["mar_impressions"]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
tr_idx2, te_idx2 = next(gss2.split(leak_df, groups=leak_df["client_hash_id"]))
train_leak, test_leak = leak_df.iloc[tr_idx2], leak_df.iloc[te_idx2]

scaler_leak = StandardScaler()
X_train_leak = scaler_leak.fit_transform(train_leak[LEAKY_FEATURE_COLS])
X_test_leak = scaler_leak.transform(test_leak[LEAKY_FEATURE_COLS])
logreg_leak = LogisticRegression(random_state=RANDOM_SEED, max_iter=1000).fit(
    X_train_leak, train_leak[LABEL_COL].values
)
leaky_scores = logreg_leak.predict_proba(X_test_leak)[:, 1]
leaky_p20 = precision_at_k(leaky_scores, test_leak[LABEL_COL].values, 20)

print(f"WITH the leaky mar_impressions feature -- precision@20: {leaky_p20:.1%}")
print(f"WITHOUT it (honest model from Section 2, grouped split) -- precision@20: {p20_g:.1%}")
print(f"Collapse from leaky to honest: {(leaky_p20 - p20_g)*100:.1f} points")
print("-> This IS the confession the skill describes: a collapse from near-1.0 back down to the")
print("   honest number confirms the test harness correctly detects leakage when it's injected --")
print("   which also means the honest number above was never inflated by it in the first place.")

print("\n== Static column-set check (unchanged from w04/w05) ==")
score_inputs = set(FEATURE_COLS)
label_or_future_cols = {"declined_next_month", "mar_impressions"}
print("Score inputs touching label/future (should be empty):", score_inputs & label_or_future_cols or "none")
product_flag_cols = {"health_score", "priority_score", "action_type", "refresh_tier"}
print("FlyRank product-decision columns present (should be empty):", product_flag_cols & set(model_df.columns) or "none")

WITH the leaky mar_impressions feature -- precision@20: 100.0%
WITHOUT it (honest model from Section 2, grouped split) -- precision@20: 85.0%
Collapse from leaky to honest: 15.0 points
-> This IS the confession the skill describes: a collapse from near-1.0 back down to the
   honest number confirms the test harness correctly detects leakage when it's injected --
   which also means the honest number above was never inflated by it in the first place.

== Static column-set check (unchanged from w04/w05) ==
Score inputs touching label/future (should be empty): none
FlyRank product-decision columns present (should be empty): none


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (w04, Section 1 — my baseline rule):** *"A page is worth refreshing if it has
real traffic worth saving (visible) and it's old enough that staleness is a plausible **cause**
(stale)..."*

**The problem:** "cause" is a causal claim. All I actually did was check a correlational bucket
table (decline rate by age tier) — I never ran an experiment or had a counterfactual, so I can't
say staleness *causes* decline, only that certain age bands are *associated with* different
observed decline rates (and non-monotonically at that, per the same section's own MIXED
verdict).

**Rewrite:** *"A page is worth refreshing if it has real traffic worth saving (visible) and it
falls in the age band where decline is most often observed (stale) — an associated pattern from
the bucket check, not a demonstrated cause."*

**Second example (w05, Section 4 — feature importance write-up):** *"`declining_now` is the
strongest single signed **driver**"* -> rewritten to *"`declining_now` is the strongest single
signed **predictor** in the fitted model"* — "driver" implies the feature causes the outcome;
"predictor" states only what the correlational model actually supports.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Nothing to compute for a text rewrite -- the audit itself lives in the markdown above.
# Leaving this cell intentionally minimal is itself the honest choice: padding it with an
# unrelated calculation would be decoration, not evidence.
print("Claim rewrite complete -- see markdown above.")

Claim rewrite complete -- see markdown above.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.